In [ ]:
import requests
import re
import pandas as pd
import xml.etree.ElementTree as ET
import json
import os
import sys
import matplotlib.pyplot as plt
from collections import Counter
import xml.etree.ElementTree as ET
from typing import List, Dict, Any, Optional
import duckdb
import unicodedata
import importlib
import numpy as np
import csv

from pathlib import Path
import re
import pandas as pd

import gspread
from gspread_dataframe import get_as_dataframe, set_with_dataframe
from google.oauth2 import service_account # based on google-auth library

In [2]:
import grela_helpers

In [3]:
conn = grela_helpers.make_connection()

In [5]:
#grela_extraction_helpers.conn.close()
# importlib.reload(grela_extraction_helpers)

In [4]:
# establish connection with Google sheets
file_data = json.load(open(os.path.expanduser("../../../ServiceAccountsKey.json")))
# (2) transform the content into crendentials object
credentials = service_account.Credentials.from_service_account_info(file_data)
# (3) specify your usage of the credentials
scoped_credentials = credentials.with_scopes(
    ['https://spreadsheets.google.com/feeds', 'https://www.googleapis.com/auth/drive'])
# (4) use the constrained credentials for authentication of gspread package
gc = gspread.Client(auth=scoped_credentials)
# url = "" # put your Google Sheets URL here
# phylosen_gs = gc.open_by_url(url)

In [5]:
# test with one specific multiword phrase
target_canonical     =  "lapis philosophorum" # "lapis philosophorum"
target_relemmatized  =  "lapis philosophus" # "lapis philosophus"
# 1) phase 1: spans only (whole GreLa, but capped)
grela_helpers.create_target_spans_table(
    conn,
    table_name="temp_target_spans_table",
    target_canonical=target_canonical,
    target_relemmatized=target_relemmatized,
    emlap_only=False,
    max_hits=1_000_000,   # safeguard
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [6]:
conn.execute("SELECT COUNT(*) FROM temp_target_spans_table").fetchone()

(762,)

In [7]:
# 2) phase 2: build detailed KWIC for first N hits
df = grela_helpers.kwic_from_spans(
    conn,
    spans_table="temp_target_spans_table",
    window=10,
    include_tokens=True,
    max_hits=100_000,     # or None, if you know it's safe
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [8]:
df

,target_phrase,target_from,matched_by,target_lemmata,target_token_ids,target_kwic_idx,target_sent_idx,grela_id,target_sentence_id,start_sentence_id,end_sentence_id,kwic_text,kwic_tokens,target_sentence_text,target_sentence_tokens,author,title,not_before,not_after
0,"[Lapis, Philosophorum]",relemmatized,lemma,"[lapis, philosophus]","[255665473, 255665474]","[10, 11]","[8, 9]",emlap_100094,emlap_100094_5849,emlap_100094_5848,emlap_100094_5852,"arte . Entium opus tendit ad perfectionem , pr...","[{'token_id': 255665463, 'token_text': 'arte',...","Entium opus tendit ad perfectionem , praeparat...","[{'token_id': 255665465, 'token_text': 'Entium...",NaN,Musaeum Hermeticum,1625,1625
1,"[Lapidem, Philosophorum]",relemmatized,lemma,"[lapis, philosophus]","[259343424, 259343425]","[10, 11]","[0, 1]",emlap_100099,emlap_100099_217,emlap_100099_216,emlap_100099_217,optimis medicamentis ad de pellendos morbos id...,"[{'token_id': 259343414, 'token_text': 'optimi...",Lapidem Philosophorum seu tincturam & Medicina...,"[{'token_id': 259343424, 'token_text': 'Lapide...","Francus, Johannes",De arte chemica,1607,1607
2,"[lapidis, Philosophorum]",relemmatized,lemma,"[lapis, philosophus]","[349280851, 349280852]","[10, 11]","[2, 3]",noscemus_616642,noscemus_616642_14838,noscemus_616642_14837,noscemus_616642_14838,"taliter , ut comestum est , excernitur . Aurum...","[{'token_id': 349280841, 'token_text': 'talite...",Aurum autem lapidis Philosophorum est solum qu...,"[{'token_id': 349280849, 'token_text': 'Aurum'...","Grataroli, Guglielmo","Alchemiae, quam vocant artisque metallicae doc...",1572,1572
3,"[Lapidis, Philosophorum]",relemmatized,lemma,"[lapis, philosophus]","[291101259, 291101260]","[10, 11]","[10, 11]",noscemus_1085290,noscemus_1085290_57551,noscemus_1085290_57551,noscemus_1085290_57552,Adjecta est Ejusdem Epistola ad Regem Ro bertu...,"[{'token_id': 291101249, 'token_text': 'Adject...",Adjecta est Ejusdem Epistola ad Regem Ro bertu...,"[{'token_id': 291101249, 'token_text': 'Adject...","Linden, Johannes Antonides van der","De scriptis medicis libri duo, quibus praemitt...",1637,1637
4,"[Lapide, Philosophorum]",relemmatized,lemma,"[lapis, philosophus]","[304323709, 304323710]","[10, 11]","[6, 7]",noscemus_631363,noscemus_631363_13977,noscemus_631363_13976,noscemus_631363_13978,", in fol . Ioannis Dastin Uisio , seu de Lapid...","[{'token_id': 304323699, 'token_text': ',', 'l...","Ioannis Dastin Uisio , seu de Lapide Philosoph...","[{'token_id': 304323703, 'token_text': 'Ioanni...","Linden, Johannes Antonides van der","De scriptis medicis libri duo, quibus praemitt...",1637,1637
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
757,"[lapide, Philosophorum]",relemmatized,lemma,"[lapis, philosophus]","[350016884, 350016885]","[10, 11]","[3, 4]",noscemus_770048,noscemus_770048_8870,noscemus_770048_8868,noscemus_770048_8871,. Eifrid . 8 . 1624 . Clangor Buccinae de lapi...,"[{'token_id': 350016874, 'token_text': '.', 'l...",Clangor Buccinae de lapide Philosophorum 4 .,"[{'token_id': 350016881, 'token_text': 'Clango...","Draud, Georg",Bibliotheca classica sive Catalogus officinali...,1625,1625
758,"[lapidem, Philosophorum]",relemmatized,lemma,"[lapis, philosophus]","[257210953, 257210954]","[10, 11]","[14, 15]",emlap_100073,emlap_100073_886,emlap_100073_886,emlap_100073_887,est non reperire ignem qui totam materiam conu...,"[{'token_id': 257210943, 'token_text': 'est', ...",Error igitur istius artis est non reperire ign...,"[{'token_id': 257210939, 'token_text': 'Error'...","Greverus, Jodocus",Secretum,1599,1599
759,"[lapide, Philosophorum]",relemmatized,lemma,"[lapis, philosophus]","[261552685, 261552686]","[10, 11]","[31, 32]",emlap_100015,emlap_100015_6995,emlap_100015_6995,emlap_100015_6996,"non nego , nec omnino nihil esse , quod de lap...","[{'token_id': 261552675, 'token_text': 'non', ...",( non tamen sub forma aquae aut olei ) possit ...,"[{'token_id': 261552654, 'token_text': '(', 'l...","Gessner, Conra

In [ ]:
len(df)

In [10]:
def safe_name(s: str | None) -> str:
    """Make a safe short filename component from a target phrase."""
    if not s or not isinstance(s, str):
        return "unknown"
    s = s.strip().lower().replace(" ", "_")
    s = re.sub(r"[^a-z0-9_]+", "", s)   # keep only safe chars
    return s or "unnamed"

In [12]:
outdir = Path("../data/large_files/phylosen_instances/")
os.makedirs("../data/large_files", exist_ok=True)
os.makedirs(outdir, exist_ok=True)

In [13]:
def from_entries_to_file(tc, tr, outdir):
    name = safe_name(tc or tr)
    fpath = str(outdir / f"{name}.parquet")

    grela_helpers.create_target_spans_table(
        conn,
        table_name="temp_target_spans_table",
        target_canonical=tc,
        target_relemmatized=tr,
        emlap_only=False,
        max_hits=1_000_000,  # safeguard
    )

    # kwic_from_spans now batches automatically when out_path is set
    grela_helpers.kwic_from_spans(
        conn,
        spans_table="temp_target_spans_table",
        window=10,
        include_tokens=True,
        out_path=fpath,
        # optional: pass through tuning knobs if you exposed them
        batch_threshold=100_000,
        batch_size=10_000,
    )

In [16]:
case_studies_terms = ["seminarium", "mercurius", "labyrinthus"]

In [14]:
for tc in case_studies_terms:
    from_entries_to_file(tc, None, outdir)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [19]:
def read_hits(fname):
    try:
        path = os.path.join(outdir, fname)
        instances = pd.read_parquet(path)
    except:
        instances = pd.DataFrame()
    return instances

In [31]:
df = read_hits("mercurius.parquet")

In [32]:
df.sample(10)

,target_phrase,target_from,matched_by,target_lemmata,target_token_ids,target_kwic_idx,target_sent_idx,grela_id,target_sentence_id,start_sentence_id,end_sentence_id,kwic_text,kwic_tokens,target_sentence_text,target_sentence_tokens,author,title,not_before,not_after
24510,[mercurius],canonical,lemma,[mercurius],[256534085],[10],[4],emlap_100085,emlap_100085_4945,emlap_100085_4944,emlap_100085_4946,"absurda , ut nihil absurdius . Sal , sulphur ,...","[{'token_id': 256534075, 'token_text': 'absurd...","Sal , sulphur , mercurius Chymicorum non possu...","[{'token_id': 256534081, 'token_text': 'Sal', ...","Libavius, Andreas",Commentariorum alchymiae,1606.0,1606.0
33693,[Mercurii],canonical,lemma,[mercurius],[396881677],[10],[4],noscemus_605919,noscemus_605919_13659,noscemus_605919_13658,noscemus_605919_13662,. 161 27 . 30 . animaduersis semper altitudini...,"[{'token_id': 396881667, 'token_text': '.', 'l...",animaduersis semper altitudinibus & Mercurii &...,"[{'token_id': 396881673, 'token_text': 'animad...","Scheuchzer, Johann Jakob","Acarnania sive Relatio eorum, quae hactenus el...",1723.0,1723.0
13596,[Mercurii],canonical,lemma,[mercurius],[291378511],[10],[39],noscemus_906184,noscemus_906184_8008,noscemus_906184_8008,noscemus_906184_8009,cum constitui Thermometra in proximis experime...,"[{'token_id': 291378501, 'token_text': 'cum', ...","congelatio Mercurii , tanquam ueritas quam uer...","[{'token_id': 291378472, 'token_text': 'congel...",Anonymus 1,[Review of:] Iohannis Alberti Euler Academiae ...,1761.0,1761.0
5405,[Mercurii],canonical,lemma,[mercurius],[364175690],[10],[8],noscemus_668501,noscemus_668501_387,noscemus_668501_386,noscemus_668501_387,opacitas . Hinc enim omnino mirari subit quo p...,"[{'token_id': 364175680, 'token_text': 'opacit...",Hinc enim omnino mirari subit quo pacto tantul...,"[{'token_id': 364175682, 'token_text': 'Hinc',...","Hortensius, Martinus",Martini Hortensi Delfensis dissertatio de Merc...,1633.0,1633.0
13666,[Mercurio],canonical,lemma,[mercurius],[281127878],[10],[6],noscemus_725077,noscemus_725077_18051,noscemus_725077_18050,noscemus_725077_18052,"nomen erat Iohannet : hic uenit in tertini , i...","[{'token_id': 281127868, 'token_text': 'nomen'...","hic uenit in tertini , i Mercurio Elias , Luna...","[{'token_id': 281127872, 'token_text': 'hic', ...","Cellarius, Andreas",Harmonia macrocosmica seu Atlas universalis et...,1661.0,1661.0
23493,[Mercurio],canonical,lemma,[mercurius],[335114103],[10],[8],noscemus_617658,noscemus_617658_2364,noscemus_617658_2363,noscemus_617658_2364,"corrigendo . Sequente die 4 Maji , cum Sol sti...","[{'token_id': 335114093, 'token_text': 'corrig...","Sequente die 4 Maji , cum Sol stipatus Mercuri...","[{'token_id': 335114095, 'token_text': 'Sequen...","Hevelius, Johannes",Mercurius in sole visus Gedani anno Christiano...,1662.0,1662.0
2478,[Mercurio],canonical,lemma,[mercurius],[388420725],[10],[25],noscemus_833197,noscemus_833197_13444,noscemus_833197_13444,noscemus_833197_13445,", minor adhuc ex Uenere , omnium que minima ex...","[{'token_id': 388420715, 'token_text': ',', 'l...",Curuatura haec paulo minor erit ex Joue uel Ma...,"[{'token_id': 388420700, 'token_text': 'Curuat...","Gregory, David",Astronomiae physicae et geometricae elementa. ...,1702.0,1702.0
9388,[Mercurius],canonical,lemma,[mercurius],[367166269],[10],[3],noscemus_906162,noscemus_906162_11017,noscemus_906162_11015,noscemus_906162_11019,. latit . 37 ' Uen . B. 19 3 Mercurius 17b in ...,"[{'token_id': 367166259, 'token_text': '.', 'l...",B. 19 3 Mercurius 17b in conj .,"[{'token_id': 367166266, 'token_text': 'B.', '...","Caturegli, Pietro",Ephemerides motuum caelestium ex anno 1829 in ...,1828.0,1828.0
27151,[Mercurio],canonical,lemma,[mercurius],[257580594],[10],[44],emlap_100038,emlap_100038_7927,emlap_100038_7927,emlap_100038_7928,corpori imperfecto mortuo uitam tribuit median...,"[{'token_id': 257580584, 'token_text': 'corpor...",Et quia sine fermentonulla fit tinctura perfec...,"[{'token_id'

In [33]:
len(df)

34560

In [37]:
len(set(df["target_token_ids"].apply(lambda x: x[0])))

34560

In [40]:
df.sample(10)

,target_phrase,target_from,matched_by,target_lemmata,target_token_ids,target_kwic_idx,target_sent_idx,grela_id,target_sentence_id,start_sentence_id,end_sentence_id,kwic_text,kwic_tokens,target_sentence_text,target_sentence_tokens,author,title,not_before,not_after
16370,[Mercurio],canonical,lemma,[mercurius],[258529192],[10],[1],emlap_100065,emlap_100065_7943,emlap_100065_7941,emlap_100065_7944,"uitriolo Romano , Ungaricoue ; non Ungarico An...","[{'token_id': 258529182, 'token_text': 'uitrio...",non Mercurio Carnithio ;,"[{'token_id': 258529191, 'token_text': 'non', ...","Libavius, Andreas",Neoparacelsica,1594.0,1594.0
14659,[Mercurii],canonical,lemma,[mercurius],[86857867],[10],[126],cc_13007,cc_13007_1816,cc_13007_1816,cc_13007_1816,auctor infra in capitulo XI° sed opus sibi fui...,"[{'token_id': 86857857, 'token_text': 'auctor'...",Post hec videamus quid secundario auctor alleg...,"[{'token_id': 86857741, 'token_text': 'Post', ...",Pietro Alighieri,Comentum super poema Comedie Dantis (III red.),1300.0,1364.0
26335,[Mercurio],canonical,lemma,[mercurius],[3325164],[10],[1],cc_12194,cc_12194_2574,cc_12194_2573,cc_12194_2575,+ additi enim coniunctione multiplici genere d...,"[{'token_id': 3325154, 'token_text': '+', 'lem...",A Mercurio Luna ad Solem .,"[{'token_id': 3325163, 'token_text': 'A', 'lem...",Julius Firmicus Maternus,Mathesis,337.0,337.0
34428,[Mercurius],canonical,lemma,[mercurius],[280821956],[10],[3],noscemus_725077,noscemus_725077_1466,noscemus_725077_1465,noscemus_725077_1466,"jacturam faciant , & arlac perlarum : Coralatu...","[{'token_id': 280821946, 'token_text': 'jactur...","Coralatus , seu Mercurius pur tem ipsam prosti...","[{'token_id': 280821953, 'token_text': 'Corala...","Cellarius, Andreas",Harmonia macrocosmica seu Atlas universalis et...,1661.0,1661.0
7791,[Mercurius],canonical,lemma,[mercurius],[275438410],[10],[4],noscemus_699869,noscemus_699869_61977,noscemus_699869_61975,noscemus_699869_61978,. Scholio 3 Concaudistantia Ii . Quod autem Ue...,"[{'token_id': 275438400, 'token_text': '.', 'l...",Quod autem Uenus ac Mercurius in hypothesi Pto...,"[{'token_id': 275438406, 'token_text': 'Quod',...","Riccioli, Giovanni Battista",Almagestum Novum astronomiam veterem novamque ...,1651.0,1665.0
29053,[Mercurium],canonical,lemma,[mercurius],[259689569],[10],[3],emlap_100100,emlap_100100_20103,emlap_100100_20101,emlap_100100_20103,fiendi . Prout suo capite planius . Paracelsus...,"[{'token_id': 259689559, 'token_text': 'fiendi...",Paracelsus autem uoluit Mercurium liquori prae...,"[{'token_id': 259689566, 'token_text': 'Parace...","Van Helmont, Jan Baptist",Ortus medicinae,1648.0,1648.0
26278,[mercurii],canonical,lemma,[mercurius],[349102559],[10],[19],noscemus_616642,noscemus_616642_5157,noscemus_616642_5157,noscemus_616642_5158,"indiges , cum ex eis habere non possis substan...","[{'token_id': 349102549, 'token_text': 'indige...","Licet aliis metallis uti possis , tamen eius n...","[{'token_id': 349102540, 'token_text': 'Licet'...","Grataroli, Guglielmo","Alchemiae, quam vocant artisque metallicae doc...",1572.0,1572.0
20988,[Mercurii],canonical,lemma,[mercurius],[324316236],[10],[1],noscemus_798546,noscemus_798546_4888,noscemus_798546_4886,noscemus_798546_4889,"Anni . Astronomi , tum Ptolemaici tum Copernic...","[{'token_id': 324316226, 'token_text': 'Anni',...",- Mercurii 23 . 87 . 13 .,"[{'token_id': 324316235, 'token_text': '-', 'l...","Scheuchzer, Johann Jakob","Physica sacra Iohannis Iacobi Scheuchzeri, med...",1731.0,1735.0
26938,[Mercurii],canonical,lemma,[mercurius],[275447847],[10],[3],noscemus_699869,noscemus_699869_62553,noscemus_699869_62552,noscemus_699869_62556,Longomontanica Hy & post 3 periodos . pothesi ...,"[{'token_id': 275447837, 'token_text': 'Longom...",pothesi Ueneris ac Mercurii Eccentrici Apogei ...,"[{'token_id': 275447844, 'token_text': 'pothes...","Riccioli, Giovanni Battista",Almagestum Novum astronomiam veterem novamque ...,1651.0,1665.0
32931,[Mercurio],canonical,lemma,[mercu